# 02 — Correlation Analysis

**Primary:** Tianyi Qin  
**Support:** Tuan Wei

Requirement: apply **Pearson, Spearman, Mutual Information (MI), and Normalised Mutual Information (NMI)** to every pair in the designed variable set, including the target or a suitable target representation.

## 1. Imports and processed data

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mutual_info_score, normalized_mutual_info_score
from sklearn.preprocessing import KBinsDiscretizer

candidates = [Path("../data/processed_listings.csv"), Path("data/processed_listings.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("Create data/processed_listings.csv from preprocessing first.")

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)

## 2. Define the RQ-driven variable set

Replace placeholders only with agreed processed columns. Include property-size variables plus location/amenity variables and the target/proxy.

In [ ]:
VARIABLES = [
    # Example only:
    # "accommodates",
    # "bedrooms",
    # "bathrooms_num",
    # "distance_cbd_km",
    # "amenity_count",
    # "high_price",
]

if VARIABLES:
    missing = [c for c in VARIABLES if c not in df.columns]
    if missing:
        raise KeyError(f"Missing agreed columns: {missing}")
    display(df[VARIABLES].describe(include="all"))
else:
    print("TODO: define the final variable set.")

## 3. Declare variable types

This is needed because not every coefficient is meaningful for every type.

In [ ]:
VARIABLE_TYPES = {
    # "accommodates": "continuous",
    # "high_price": "binary",
    # "neighbourhood_group": "categorical",
}
VARIABLE_TYPES

## 4. Helpers for MI/NMI

MI/NMI on continuous variables needs an explicit discretisation choice if using `mutual_info_score` / `normalized_mutual_info_score`. The helper below uses quantile bins and records the choice. If the group chooses another implementation, replace it consistently.

In [ ]:
def discretize_for_information(series, n_bins=5):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(index=series.index, dtype="float64")

    valid = s.notna()
    if valid.sum() < 2:
        return out

    # duplicates='drop' handles repeated quantile edges.
    out.loc[valid] = pd.qcut(
        s.loc[valid],
        q=n_bins,
        labels=False,
        duplicates="drop"
    )
    return out

def as_discrete(series, declared_type, n_bins=5):
    if declared_type in {"categorical", "binary", "ordinal_discrete"}:
        return series.astype("category").cat.codes.replace(-1, np.nan)
    return discretize_for_information(series, n_bins=n_bins)

## 5. Compute every pair

The output deliberately includes an `appropriateness_note` so methods are not applied silently where they are not defensible.

In [ ]:
rows = []

for a, b in combinations(VARIABLES, 2):
    pair = df[[a, b]].dropna().copy()
    type_a = VARIABLE_TYPES.get(a, "unspecified")
    type_b = VARIABLE_TYPES.get(b, "unspecified")

    row = {
        "var_a": a,
        "var_b": b,
        "n": len(pair),
        "pearson": np.nan,
        "spearman": np.nan,
        "mi": np.nan,
        "nmi": np.nan,
        "appropriateness_note": "",
    }

    both_numeric = (
        pd.api.types.is_numeric_dtype(pair[a])
        and pd.api.types.is_numeric_dtype(pair[b])
    )

    if len(pair) > 1 and both_numeric:
        row["pearson"] = pearsonr(pair[a], pair[b]).statistic
        row["spearman"] = spearmanr(pair[a], pair[b]).statistic
    else:
        row["appropriateness_note"] += "Pearson/Spearman not computed: non-numeric representation. "

    if len(pair) > 1:
        da = as_discrete(pair[a], type_a)
        db = as_discrete(pair[b], type_b)
        valid = da.notna() & db.notna()

        if valid.sum() > 1:
            row["mi"] = mutual_info_score(da.loc[valid], db.loc[valid])
            row["nmi"] = normalized_mutual_info_score(
                da.loc[valid], db.loc[valid]
            )

    rows.append(row)

corr_results = pd.DataFrame(rows)
display(corr_results)

## 6. Required evidence to preserve

Do not write generic conclusions here. Record actual values needed for the group-written report:
- strongest, weakest, and absent relationships;
- where methods agree;
- where they diverge and a reason grounded in the data;
- predictor–predictor associations relevant to multicollinearity;
- one downstream modelling/feature decision that changed because of these results;
- plausible confounders/biases using non-causal language.

In [ ]:
# Optional export for report table preparation
# out = Path("../output/tables/correlation_results.csv") if Path("../output/tables").exists() else Path("output/tables/correlation_results.csv")
# corr_results.to_csv(out, index=False)
# print("Saved:", out.resolve())